# 1 - Setup

In [ ]:
%%capture
%pip install --quiet pydantic-ai nest_asyncio scikit-learn matplotlib umap-learn
import nest_asyncio
nest_asyncio.apply() #Jupyter notebook-specific fix

In [ ]:
#TODO Set your OpenRouter API key
import os
os.environ['OPENROUTER_API_KEY'] = 'YOUR_KEY_GOES_HERE'

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

MODEL = OpenAIChatModel(
    'anthropic/claude-haiku-4.5',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
)

In [ ]:
def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)
            print(f'    └─ {kind}: {snippet}')

In [29]:
!rm -rf ./data
!mkdir -p ./data/sc_expression ./data/dna_seq

!wget -q -P ./data/sc_expression https://raw.githubusercontent.com/MartinekV/ismb-agentic-tutorial/main/data/sc_expression/synthetic_expression.csv
!wget -q -P ./data/sc_expression https://raw.githubusercontent.com/MartinekV/ismb-agentic-tutorial/main/data/sc_expression/synthetic_metadata.csv

!wget -q -P ./data/dna_seq https://raw.githubusercontent.com/MartinekV/ismb-agentic-tutorial/main/data/dna_seq/experiment_1.csv
!wget -q -P ./data/dna_seq https://raw.githubusercontent.com/MartinekV/ismb-agentic-tutorial/main/data/dna_seq/experiment_2.csv

# 2 - Multiple agents
Various ways to use multiple agents - for more info explore pydantic-AI docs
https://pydantic.dev/docs/ai/guides/multi-agent-applications/

In [ ]:
from pathlib import Path
from pprint import pprint
import pandas as pd
from pydantic_ai import Agent, RunContext, ModelRetry
from pydantic import BaseModel, Field

agent_a = Agent(
    MODEL,
    output_type=int,
    system_prompt="Pick a random number",
)

agent_b = Agent(
    MODEL,
    deps_type=int,
    output_type=bool,
)

@agent_b.system_prompt
def system_prompt(ctx: RunContext[int]) -> str:
    return f"Your job is to judge whether {ctx.deps} is even or not"

agent_a_result = agent_a.run_sync()
agent_b_result = agent_b.run_sync(deps=agent_a_result.output)

pretty_print_trace(agent_a_result)
print('-'*50)
pretty_print_trace(agent_b_result)
print('-'*50)
print(agent_b_result.output)

## 2.1

In [ ]:
class ScriptSpec(BaseModel):
    input_args : list[tuple[str,str]] = Field(
        description=
            "List of tuples containing arguments required to the script (name and description). " \
            "E.g. [('--input-csv', 'csv with columns [sequence, score] where sequence is str and score is float')," \
            "('--output', 'the path to the output png file')]"
    )
    strategy: str = Field(
        description=
            "Description on what the script should implement and how it should use its arguments."
        )
    arg_to_file_map: dict[str,str] = Field( #only for validation
        description=
            "Mapping of all input arguments to existing files. E.g. {'--input-csv': 'input.csv', '--output': 'output.png'}"
    )

def list_dir(ctx: RunContext[Path]):
    return list(Path(ctx.deps).iterdir())

def get_csv_head(ctx: RunContext[Path], csv_path:str):
    if ctx.deps not in Path(csv_path).absolute().parts:
        return f'Path {csv_path} is not in the folder {ctx.deps}'
    return pd.read_csv(csv_path).head().to_dict()

spec_maker = Agent(
    MODEL,
    tools=[get_csv_head, list_dir],
    deps_type=Path, #Path that the agent can access with the tools
    output_type=ScriptSpec,
    system_prompt="Write a spec for a script specified by the user. The script may use files from the available directory.",
)

spec_result = spec_maker.run_sync(
    user_prompt=
        "Create a clustering script (1 cluster per cell type) based on the gene experssion, then make a visualization (plots only) " \
        "called clusters_analysis.png that will show if our cell type annotation makes good clusters.",
    deps=Path('data/sc_expression'))

pprint(spec_result.output.model_dump())

## 2.2

In [ ]:
import subprocess

def write_python_script(filename: str, code: str) -> str:
    DATA_DIR = Path('./agent_scripts')
    DATA_DIR.mkdir(exist_ok=True)
    script_path = DATA_DIR / filename
    script_path.write_text(code, encoding='utf-8')
    return f'Script written to: {script_path}'

coding_agent = Agent(
    MODEL, 
    tools=[write_python_script], #coder agent will never see the data -> no worry about leaking data
    deps_type=ScriptSpec,
    output_type=Path,
)

@coding_agent.system_prompt  
async def get_system_prompt(ctx: RunContext[ScriptSpec]) -> str:  
  return (
     f"You are a coding agent that will write a script." 
     f"The required arguments: {ctx.deps.input_args}."
     f"The functionality description: {ctx.deps.strategy}."
     f"Return only the path to the script."
  )

@coding_agent.output_validator
def validate_script_runs(ctx: RunContext[ScriptSpec], script_path: Path) -> Path:
    args = [item for pair in ctx.deps.arg_to_file_map.items() for item in pair]
    result = subprocess.run(['python', str(script_path), *args], 
        shell=False,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise ModelRetry(f"The script failed to run. ({result.stderr})")
    return script_path

## 2.3

In [ ]:
spec_result = spec_maker.run_sync(
    user_prompt=
        "Create a clustering script (1 cluster per cell type) based on the gene experssion, then make a visualization (plots only) " \
        "called clusters_analysis.png that will show if our cell type annotation makes good clusters.",
    deps=Path('data/sc_expression')
)
coding_result = coding_agent.run_sync(
    deps=spec_result.output, 
    retries=2
)

# Are there any mislabeled cells?
print(coding_result.output)

## 2.4

In [ ]:
spec_result = spec_maker.run_sync(
    user_prompt=
        "Create a script showing the distribution differences in the various experiments, then make a " \
        "visualization called dna_distr_analysis.png that will show it. Make sure to include sequence length distributions.",
    deps=Path('data/dna_seq')
)
coding_result = coding_agent.run_sync(
    deps=spec_result.output, 
    retries=2
)
print(coding_result.output)

## Exercise 1

In [ ]:
# TODO Add a critique agent that will review a code file and tell the user if it's a) correct and b) well-structured
